# 01 — Generative Model Landscape

A brief history of how generative modelling evolved from VAEs and GANs to today's diffusion-dominant paradigm.

## VAE, GAN, Flow, and Diffusion — Why Diffusion Won

**Variational Autoencoders (VAEs, Kingma & Welling 2013)** encode data into a continuous latent space and decode samples. Training is stable (ELBO objective), but the Gaussian posterior assumption causes blurry samples because the decoder averages over modes. Covered in depth in Series 9.

**Generative Adversarial Networks (GANs, Goodfellow 2014)** train a generator and discriminator in a minimax game. Sharp, high-fidelity samples — but training is famously unstable:
- *Mode collapse*: generator finds a few modes that fool the discriminator and ignores the rest
- *Training dynamics*: the discriminator can overpower the generator early, causing vanishing gradients
- DCGAN, WGAN, StyleGAN each addressed different failure modes. By ~2022, StyleGAN2 produced state-of-the-art faces, but samples lacked text coherence and generalization was limited.

**Normalising Flows (NICE, Real NVP, Glow)**: model an exact likelihood using invertible transformations. Exact density estimation and exact sampling — but invertibility constrains architecture expressiveness, and Glow's quality on complex data was outpaced by other methods by 2021.

**Diffusion models (DDPM, Ho et al. 2020)** sidestep all these problems:
- No adversarial training — stable, reproducible training
- No invertibility constraint — deep UNets fine
- No posterior blurriness — sampling iteratively refines
- Scales to arbitrary data types (images, audio, protein structure, video)

In [ ]:
# VAE recap — pointer to Series 9
# This cell demonstrates the core VAE limitation: posterior blurriness
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Simple 2D latent space demo
class TinyVAE(nn.Module):
    def __init__(self, input_dim=2, latent_dim=2, hidden=32):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU())
        self.mu_head = nn.Linear(hidden, latent_dim)
        self.logvar_head = nn.Linear(hidden, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.mu_head(h), self.logvar_head(h)

    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        return self.decoder(z), mu, logvar

# Two-mode dataset
np.random.seed(42)
mode1 = np.random.randn(200, 2) * 0.3 + np.array([2, 2])
mode2 = np.random.randn(200, 2) * 0.3 + np.array([-2, -2])
data = torch.tensor(np.vstack([mode1, mode2]), dtype=torch.float32)

vae = TinyVAE()
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
for _ in range(300):
    recon, mu, logvar = vae(data)
    recon_loss = ((recon - data)**2).mean()
    kl_loss = -0.5 * (1 + logvar - mu**2 - logvar.exp()).mean()
    loss = recon_loss + 0.1 * kl_loss
    opt.zero_grad(); loss.backward(); opt.step()

# Sample from VAE
with torch.no_grad():
    z_samples = torch.randn(200, 2)
    vae_samples = vae.decoder(z_samples).numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(data[:, 0], data[:, 1], alpha=0.3, s=5, label='Real')
ax1.set_title('Real data (two modes)')
ax2.scatter(vae_samples[:, 0], vae_samples[:, 1], alpha=0.3, s=5, color='orange', label='VAE')
ax2.set_title('VAE samples (blurry — fills inter-mode space)')
for ax in [ax1, ax2]: ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
plt.suptitle('VAE posterior blurriness on bimodal data')
plt.tight_layout()
plt.savefig('/tmp/vae_blurry.png', dpi=100, bbox_inches='tight')
plt.show()
print('VAE blurriness demo saved')